In [1]:
import pandas as pd

In [2]:
processed = pd.read_csv("../data/processed/team_stats_processed.csv")
future = pd.read_csv("../data/raw/future_games.csv", index_col=0)
print(processed.shape, future.shape)

(3230, 86) (272, 46)


In [3]:
SCHED_KEEP = [
    "season", "week", "game_type",
    "away_team", "home_team",
    "away_rest", "home_rest", "div_game", "roof", "surface", "temp", "wind",
]

upcoming = future[future["game_type"] == "REG"][SCHED_KEEP].copy()
print(f"{len(upcoming)} upcoming games")
upcoming.head()

272 upcoming games


,season,week,game_type,away_team,home_team,away_rest,home_rest,div_game,roof,surface,temp,wind
0,2026,1,REG,NE,SEA,7,7,0,outdoors,fieldturf,NaN,NaN
1,2026,1,REG,SF,LA,7,7,1,dome,matrixturf,NaN,NaN
2,2026,1,REG,CHI,CAR,7,7,0,outdoors,grass,NaN,NaN
3,2026,1,REG,TB,CIN,7,7,0,outdoors,fieldturf,NaN,NaN
4,2026,1,REG,NO,DET,7,7,0,dome,fieldturf,NaN,NaN


In [4]:
# Melt upcoming schedule wide → team-perspective (one row per team per game)
home = upcoming.copy()
home["team"] = home["home_team"]
home["opponent_team"] = home["away_team"]
home["is_home"] = True
home["rest"] = home["home_rest"]
home["opp_rest"] = home["away_rest"]

away = upcoming.copy()
away["team"] = away["away_team"]
away["opponent_team"] = away["home_team"]
away["is_home"] = False
away["rest"] = away["away_rest"]
away["opp_rest"] = away["home_rest"]

GAME_META = ["season", "week", "team", "opponent_team",
             "is_home", "rest", "opp_rest", "div_game", "roof", "surface", "temp", "wind"]

upcoming_long = pd.concat([home[GAME_META], away[GAME_META]]).reset_index(drop=True)
upcoming_long.head()

,season,week,team,opponent_team,is_home,rest,opp_rest,div_game,roof,surface,temp,wind
0,2026,1,SEA,NE,True,7,7,0,outdoors,fieldturf,NaN,NaN
1,2026,1,LA,SF,True,7,7,1,dome,matrixturf,NaN,NaN
2,2026,1,CAR,CHI,True,7,7,0,outdoors,grass,NaN,NaN
3,2026,1,CIN,TB,True,7,7,0,outdoors,fieldturf,NaN,NaN
4,2026,1,DET,NO,True,7,7,0,dome,fieldturf,NaN,NaN


In [5]:
# Compute per-team season averages from processed data
# Use the most recent completed season as a proxy for current team strength
META_COLS = {"season", "week", "team", "opponent_team", "is_home", "won",
             "rest", "opp_rest", "overtime", "div_game", "roof", "surface", "temp", "wind"}

STAT_COLS = [c for c in processed.columns if c not in META_COLS and not c.startswith("opp_")]

latest_season = processed["season"].max()
season_avgs = (
    processed[processed["season"] == latest_season]
    .groupby("team")[STAT_COLS]
    .mean()
    .reset_index()
)

print(f"Season averages computed from {latest_season} season ({len(season_avgs)} teams)")
season_avgs.head()

Season averages computed from 2025 season (32 teams)


,team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,passing_air_yards,passing_yards_after_catch,...,penalty_yards,timeouts,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_long,fg_pct
0,ARI,25.117647,38.176471,256.117647,1.705882,0.647059,3.470588,-23.470588,274.117647,124.176471,...,49.529412,3.647059,1.117647,12.764706,4.647059,116.411765,1.470588,1.941176,44.000000,0.769231
1,ATL,19.529412,32.058824,217.823529,1.117647,0.470588,1.529412,-10.588235,245.294118,110.058824,...,48.176471,3.588235,1.411765,9.882353,3.941176,90.058824,1.882353,2.294118,45.750000,0.828431
2,BAL,16.352941,24.823529,192.823529,1.352941,0.647059,2.647059,-17.235294,201.294118,80.058824,...,43.882353,3.647059,0.941176,11.647059,4.352941,112.411765,1.764706,2.000000,38.733333,0.886667
3,BUF,20.235294,29.117647,234.176471,1.705882,0.588235,2.352941,-17.529412,212.588235,123.411765,...,49.882353,3.882353,1.764706,11.411765,3.882353,108.352941,1.117647,1.235294,41.363636,0.875000
4,CAR,19.352941,30.294118,194.352941,1.411765,0.705882,2.058824,-15.058824,202.941176,95.823529,...,44.823529,3.647059,1.411765,10.352941,3.176471,76.705882,1.411765,1.705882,41.285714,0.764706


In [6]:
# Join team averages and opponent averages to upcoming games
opp_avgs = season_avgs.rename(columns={"team": "opponent_team"})
opp_avgs.columns = ["opponent_team"] + [f"opp_{c}" for c in opp_avgs.columns if c != "opponent_team"]

pred = upcoming_long.merge(season_avgs, on="team", how="left")
pred = pred.merge(opp_avgs, on="opponent_team", how="left")

print(pred.shape)
pred.head()

(544, 84)


,season,week,team,opponent_team,is_home,rest,opp_rest,div_game,roof,surface,...,opp_penalty_yards,opp_timeouts,opp_punt_returns,opp_punt_return_yards,opp_kickoff_returns,opp_kickoff_return_yards,opp_fg_made,opp_fg_att,opp_fg_long,opp_fg_pct
0,2026,1,SEA,NE,True,7,7,0,outdoors,fieldturf,...,50.235294,3.176471,1.235294,21.352941,3.352941,85.352941,1.588235,1.882353,42.230769,0.830952
1,2026,1,LA,SF,True,7,7,1,dome,matrixturf,...,38.294118,4.117647,1.470588,17.117647,3.000000,81.470588,1.941176,2.117647,44.928571,0.928571
2,2026,1,CAR,CHI,True,7,7,0,outdoors,grass,...,57.058824,3.058824,1.235294,13.588235,4.000000,105.411765,1.941176,2.294118,40.600000,0.867778
3,2026,1,CIN,TB,True,7,7,0,outdoors,fieldturf,...,43.352941,3.823529,1.529412,17.117647,3.588235,87.294118,1.882353,2.235294,46.733333,0.817708
4,2026,1,DET,NO,True,7,7,0,dome,fieldturf,...,54.117647,3.823529,1.411765,12.882353,3.882353,97.941176,1.764706,2.470588,42.200000,0.696078


In [7]:
pred.to_csv("../data/processed/upcoming_games.csv", index=False)
print(f"Saved {len(pred)} rows, {len(pred.columns)} columns")

Saved 544 rows, 84 columns
